In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY)
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


#### Get subzone data

In [3]:

def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones = pd.read_csv(subzones_filepath)

    subzones = subzones.map(lambda s: s.lower() if isinstance(s, str) else s)
    return subzones

In [4]:
subzone_df = get_subzone_data(2014)
subzone_df.head(3)

,subzone_n,pln_area_n,beach_area,business_2,educational_institution,utility,special_use,waterbody,port_/_airport,park,...,reserve_site,white,place_of_worship,business_2_-_white,hotel,residential_/_institution,business_park_-_white,business_park,agriculture,pri_classification
0,ang mo kio town centre,ang mo kio,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,...,3618.378339,0.0,3158.691818,0.0,0.0,0.0,0.0,0.0,0.0,residential
1,cheng san,ang mo kio,0.0,0.0,69353.966728,5449.975328,0.0,0.0,0.0,0.00000,...,66034.996250,0.0,2670.252332,0.0,0.0,0.0,0.0,0.0,0.0,residential
2,chong boon,ang mo kio,0.0,0.0,100775.355941,3683.124722,0.0,0.0,0.0,30024.94591,...,0.000000,0.0,3709.155602,0.0,0.0,0.0,0.0,0.0,0.0,residential


#### Resident Density

In [5]:
def get_resident_data(year):
    """
    Args
    ------
    year: int
        year of the resident data (resident numbers and age group) you want (2020, 2015, 2010)

    Returns
    ------
    Dataframe containing the subzone names, planning area names and resident numbers and age group.
    """
    residential_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/{year}_age_group_planning_area_subzone.xlsx")

    residential_data = pd.read_excel(residential_filepath, sheet_name = "subzone")
    residential_data["planning_area"] = residential_data["planning_area"].ffill()

    return residential_data

In [6]:
def check_for_missing_subzone_pln_area(subzone_df, demographic_df):
    # check if the planning area and subzone name of both datasets match

    # 1. Perform the left join
    # subzones_2019 is the 'left' table because it contains all pairs
    check_df = pd.merge(
        subzone_df, 
        demographic_df, 
        left_on=['subzone_n', 'pln_area_n'], 
        right_on=['subzone', 'planning_area'], 
        how='left',
        indicator=True
    )

    # 2. Identify the pairs that are MISSING in residential_df
    missing_in_res = check_df[check_df['_merge'] == 'left_only']

    # 3. Identify the pairs that MATCH in both
    matching_pairs = check_df[check_df['_merge'] == 'both']

    print(f"Total pairs in Subzones df: {len(subzone_df)}")
    print(f"Matches found: {len(matching_pairs)}")
    print(f"Pairs missing in Residential df: {len(missing_in_res)}")
    if len(missing_in_res) != 0:
        print(f"Pairs missing in Residential df: {missing_in_res}")

#### Workplace classification

In [7]:
def get_landuse_dataset(year: int):
    year = str(year)

    landuse_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    landuse_df = pd.read_csv(landuse_filepath)

    return landuse_df

In [8]:
# landuse_df = get_landuse_dataset(2014)
# landuse_df

There are 3 types of workplace classification in singapore.
- business 1: Clean industry, light industry, public utilities (logistics, warehousing, technology, research)
- business 2: Special industries involving industrial machinery, shipbuilding and repairing (heavy industries)
- business park: blend of industrial and office spaces

Each of these 3 workplaces could also be classified as white sites:

Which are areas used for commercial, hotel, residential, office, recreational club. As the number of workplaces that are labelled as white sites are few, they will be combined with their respective workplace types (eg: business 1 will combine with business 1 white)

Additionally, subzones with workplace areas smaller than 1000 $m^2$ will not be included. Reason is that a workplace that is too small would not have many workers in there.

references: 
- https://propertyreviewsg.com/ura-masterplan-zoning-interpretation/
- https://www.sgindustrialgroup.com/post/the-difference-between-different-industrial-property-classes

In [9]:
def determine_workplace_type(landuse_df, columns_of_interest):

    workplace_cols = [col for col in landuse_df.columns if 'business' in col]
    columns_of_interest += workplace_cols
    workplace_df = landuse_df[columns_of_interest].copy()

    workplace_df['pln_area_n'] = workplace_df['pln_area_n'].astype(str).str.strip().str.lower()
    workplace_df['subzone_n'] = workplace_df['subzone_n'].astype(str).str.strip().str.lower()

    workplace_df[workplace_cols] = workplace_df[workplace_cols].fillna(0)

    workplace_df['pri_classification'] = workplace_df['pri_classification'].astype(str).str.strip().str.lower()

    # group and Encode (Sum of workplace areas must be > 1000, else encode as 0)
    # sum the "White" and "Non-White" variants for each category
    workplace_df["business_1_encoding"] = (
        (workplace_df['business_1'] + workplace_df['business_1_-_white']) > 1000
    ).astype(int)
    workplace_df["business_2_encoding"] = (
        (workplace_df['business_2'] + workplace_df['business_2_-_white']) > 1000
    ).astype(int)
    workplace_df["business_park_encoding"] = (
        (workplace_df['business_park'] + workplace_df['business_park_-_white']) > 1000
    ).astype(int)

    return workplace_df

In [10]:
def determine_school_airport(landuse_df, columns_of_interest):

    school_cols = [col for col in landuse_df.columns if 'education' in col]
    columns_of_interest += school_cols
    df = landuse_df[columns_of_interest].copy()

    df['pln_area_n'] = df['pln_area_n'].astype(str).str.strip().str.lower()
    df['subzone_n'] = df['subzone_n'].astype(str).str.strip().str.lower()

    df[school_cols] = df[school_cols].fillna(0)

    # group and Encode (Sum of school areas must be > 1000, else encode as 0)
    # sum the "White" and "Non-White" variants for each category
    df["school_encoding"] = (
        df['educational_institution'] > 1000
    ).astype(int)

    airport_subzones = ['seletar aerospace park', 'changi airport']

    # Set 'airport' to 1 if subzone is in the list, else 0
    df['airport'] = df['subzone_n'].isin(airport_subzones).astype(int)

    return df

In [11]:
# columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

# workplace_df = determine_workplace_type(landuse_df, columns_of_interest)
# workplace_df.head()

#### Combining all the characteristics tgt

In [12]:
# def combine_subzone_characteristics(merged_total_density_df, merged_ethnicity_density_df, workplace_df, school_df):
def combine_subzone_characteristics(workplace_df, school_df):

    subzone_combined_characteristics = workplace_df[["pln_area_n", "subzone_n", "pri_classification",
                                    "business_1_encoding", "business_2_encoding",
                                    "business_park_encoding", 'year']].copy()


    subzone_combined_characteristics = pd.merge(
        subzone_combined_characteristics,
        school_df[["pln_area_n", "subzone_n",
                                    "school_encoding", "airport"]],
        on = ["pln_area_n", "subzone_n"],
        how = "left"
    )

    return subzone_combined_characteristics

In [13]:
# subzone_combined_characteristics = combine_subzone_characteristics(merged_total_density_df, merged_ethnicity_density_df, workplace_df, education_proportion_df)
# subzone_combined_characteristics

In [14]:
# import pandas as pd

# def check_for_missing_subzone_pln_area(subzone_df, demographic_df):
#     # 1. Perform an outer join to capture mismatches from both sides
#     check_df = pd.merge(
#         subzone_df, 
#         demographic_df, 
#         left_on=['subzone_n', 'pln_area_n'], 
#         right_on=['subzone', 'planning_area'], 
#         how='outer',
#         indicator=True
#     )

#     # 2. Pairs in subzone_df but missing in demographic_df
#     missing_in_demographic = check_df[check_df['_merge'] == 'left_only']

#     # 3. Pairs in demographic_df but missing in subzone_df (What you asked for)
#     missing_in_subzone = check_df[check_df['_merge'] == 'right_only']

#     # 4. Pairs that match in both
#     matching_pairs = check_df[check_df['_merge'] == 'both']

#     print(f"Total pairs in Subzones df: {len(subzone_df)}")
#     print(f"Total pairs in Demographic df: {len(demographic_df)}")
#     print("-" * 30)
#     print(f"Matches found: {len(matching_pairs)}")
#     print(f"Pairs in Subzone df but NOT in Demographic df: {len(missing_in_demographic)}")
#     print(f"Pairs in Demographic df but NOT in Subzone df: {len(missing_in_subzone)}")

#     if not missing_in_subzone.empty:
#         print("\nPairs existing in Demographic df but missing in Subzone df:")
#         # Selecting only the demographic columns for clarity
#         print(missing_in_subzone[['subzone', 'planning_area']])

#     return missing_in_subzone # Optional: return the dataframe for further cleaning

# missing_in_subzone = check_for_missing_subzone_pln_area(subzone_df, resident_df)
# # save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/missing_pairs_from_subzone_2010.csv")
# # missing_in_subzone.to_csv(save_to_filepath)

By visually checking which subzone in 2008 overlaps with 2014 subzones, I have mapped the 2014 subzone names to the 2008 ones. 

In [15]:
# mapping_data = [
#     ("jurong east", "boon lay", "yuhua west"),
#     ("jurong west", "boon lay", "boon lay place"),
#     ("sengkang", "buangkok", "anchorvale"),
#     ("bukit panjang", "bukit panjang", "jelebu"),
#     ("choa chu kang", "central", "choa chu kang central"),
#     ("jurong west", "central", "jurong west central"),
#     ("pasir ris", "elias", "pasir ris west"),
#     ("bukit batok", "hong kah", "hong kah north"),
#     ("sengkang", "jalan kayu east", "fernvale"),
#     ("sengkang", "jalan kayu west", "sengkang west"),
#     ("jurong east", "jurong lake", "lakeside"),
#     ("jurong east", "jurong regional centre", "jurong gateway"),
#     ("toa payoh", "kallang", "lorong 8 toa payoh"),
#     ("choa chu kang", "kranji north", "choa chu kang north"),
#     ("toa payoh", "marymount", "toa payoh west"),
#     ("choa chu kang", "pang sua", "choa chu kang north"),
#     ("pasir ris", "pasir ris town", "pasir ris central"),
#     ("toa payoh", "paya lebar", "joo seng"),
#     ("hougang", "rosyth", "kovan"),
#     ("ang mo kio", "sindo", "tagore"),
#     ("punggol", "subzone 2", "punggol town centre"),
#     ("punggol", "subzone 3", "matilda"),
#     ("punggol", "subzone 4", "punggol field"),
#     ("punggol", "subzone 5", "waterway east"),
#     ("sengkang", "sungei serangoon west", "rivervale"),
#     ("hougang", "tai keng", "tai seng"),
#     ("ang mo kio", "town centre", "ang mo kio town centre"),
#     ("sengkang", "trafalgar", "compassvale"),
#     ("jurong east", "yuhua", "yuhua east")
# ]

In [16]:
# def get_subzone_densities(mapping_data):
#     demographic_subzone_mapping = {2010: 2008,
#                                    2015: 2014,
#                                    2020: 2019}
#     output = []
#     for demographic_year, subzone_year in demographic_subzone_mapping.items():
#         current_subzone_year = subzone_year
#         resident_df = get_resident_data(demographic_year)
#         # there is no landuse data for the 2008 masterplan, 
#         # so I will be using the landuse data from 2014
#         if subzone_year == 2008:
#             current_subzone_year = 2014
            
#             map_df = pd.DataFrame(mapping_data, columns=['planning_area', 'subzone', 'new_subzone'])
#             # renaming certain subzone names so they would match with subzone names from 2014
#             # Merge this with your resident_df to update the names
#             resident_df = resident_df.merge(
#                 map_df, 
#                 on=['planning_area', 'subzone'], 
#                 how='left'
#             )

#             # Replace the old subzone with the new one where a match was found
#             resident_df['subzone'] = resident_df['new_subzone'].fillna(resident_df['subzone'])
#             resident_df.drop(columns=['new_subzone'], inplace=True)

#         subzone_df = get_subzone_data(current_subzone_year)
#         print(f"checking for subzones for the year {subzone_year} for demographic data from {demographic_year}")
#         check_for_missing_subzone_pln_area(subzone_df, resident_df)

#         merged_total_density_df = calculate_density(subzone_df, resident_df)
#         # add year column for easier identification of rows
#         merged_total_density_df['year'] = demographic_year

#         merged_total_density_df["density_per_ha"] = merged_total_density_df["density_per_ha"].fillna(0)
#         df = merged_total_density_df.sort_values("density_per_ha", ascending = False)
#         print("Displaying 10 most dense subzones by resident count:")
#         display(df[['pln_area_n', 'subzone_n', 'total', 'total_residential_area', 'density_per_ha']].head(10))
        
#         output.append(merged_total_density_df)
    
#     return output
        

In [17]:
# # The table outputs should only contain subzones with the name total
# # The 3 years of dataset should have 191, 323, and 332 rows from 2010 to 2020 respectively
# subzone_resident_densities = get_subzone_densities(mapping_data)

In [18]:
def encode_workplace():
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    
    output = []
    for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
        current_subzone_year = subzone_year
        # 2008 workplace data will be from 2014 masterplan
        if subzone_year == 2008:
            current_subzone_year = 2014
        landuse_df = get_landuse_dataset(current_subzone_year)

        columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

        workplace_df = determine_workplace_type(landuse_df, columns_of_interest)
        workplace_df['year'] = demographic_year
        output.append(workplace_df)

    return output

In [19]:
encoded_workplace = encode_workplace()
encoded_workplace[1]

,subzone_n,pln_area_n,pri_classification,business_2,business_1_-_white,business_1,business_2_-_white,business_park_-_white,business_park,business_1_encoding,business_2_encoding,business_park_encoding,year
0,ang mo kio town centre,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
1,cheng san,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
2,chong boon,ang mo kio,residential,0.0,0.0,161265.128547,0.0,0.0,0.0,1,0,0,2015
3,kebun bahru,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
4,sembawang hills,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,springleaf,yishun,reserve site,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
319,yishun central,yishun,reserve site,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
320,yishun east,yishun,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
321,yishun south,yishun,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015


In [20]:
def encode_schools_airport():
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    
    output = []
    for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
        current_subzone_year = subzone_year
        # 2008 workplace data will be from 2014 masterplan
        if subzone_year == 2008:
            current_subzone_year = 2014
        landuse_df = get_landuse_dataset(current_subzone_year)

        columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

        school_airport_df = determine_school_airport(landuse_df, columns_of_interest)
        school_airport_df['year'] = demographic_year
        output.append(school_airport_df)

    return output

In [21]:
encoded_school_airport = encode_schools_airport()
# encoded_school_airport[1]

In [22]:
demographic_subzone_mapping = {2010: 2008,
                                2015: 2014,
                                2020: 2019}

for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
    subzone_combined_characteristics = combine_subzone_characteristics(encoded_workplace[i], encoded_school_airport[i])
    
    save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_combined_characteristics_{demographic_year}.csv")
    subzone_combined_characteristics.to_csv(save_to_filepath)
